# Milestone 1 lab, notebook 4 of 8: Who is hurt

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and five green checks | Meet Lotus Retails and the Monday question. Run the given pipeline for one night. Read its five checks |
| 2. Airflow runs it | Let Airflow run the same night. Read the DAG: its tasks and its arrows |
| 3. The number by hand | Add up one store's receipts yourself. Find out why the two numbers differ |
| **4. Who is hurt** (you are here) | For each table: who uses it, and what happens to them when it is wrong |
| 5. The check card | Meet the page you fill in before any check. Say what one row means |
| 6. Read the table first | Count the rows, find the smallest and largest values, count the empty cells |
| 7. Six questions for any table | Sort your worries under six questions. See which ones the five checks answer |
| 8. The production version | The same logic as a tested program |

**About ten minutes.**

In [1]:
import re

import pandas as pd

from lab_helpers import sql
from lotus_pipeline import clean, consumers, publish

pd.set_option("display.max_colwidth", 90)

## Step 1: The question nobody asked

You found a wrong number. The quick answer is to write a check for it and move on.

Stop first. Lotus has seven tables, and you could write a hundred checks. Which ones matter? A check exists to protect a person. So ask, for every table: **who uses it, and what happens to them when it is wrong?**

Start with the tables. Which one is built from which? The pipeline's own SQL says so.

In [2]:
def tables_read(statement):
    """The tables a CREATE TABLE ... AS statement reads from."""
    return sorted(set(re.findall(r"\b(?:raw|clean)\.[a-z_]+", statement.split(" AS", 1)[1])))


for name, statement in clean.STEPS.items():
    print(f"clean.{name:<16} is built from  {', '.join(tables_read(statement))}")

print()
print("output.daily_store_sales  is built from ", ", ".join(tables_read(publish.DAILY_STORE_SALES)))
in_list = sorted(set(re.findall(r"\bclean\.[a-z_]+", publish.REPLENISHMENT_LIST)))
print("output.replenishment_list is built from ", ", ".join(in_list))

clean.stores           is built from  raw.stores
clean.products         is built from  raw.products
clean.members          is built from  raw.members
clean.till_totals      is built from  raw.till_totals
clean.stock_counts     is built from  raw.stock_counts
clean.supplier_prices  is built from  raw.supplier_prices
clean.sales_lines      is built from  clean.products, clean.stores, raw.sales_lines

output.daily_store_sales  is built from  clean.sales_lines
output.replenishment_list is built from  clean.products, clean.sales_lines, clean.stock_counts, clean.stores, clean.supplier_prices


Draw that by hand. One arrow means "is built from".

```
raw.sales_lines ──┐
raw.products ─────┼─> clean.sales_lines ──┬──> output.daily_store_sales
raw.stores ───────┘                       │
                                          ├──> output.replenishment_list
raw.stock_counts ─────> clean.stock_counts ┤
raw.supplier_prices ──> clean.supplier_prices ┘

raw.members ──────────> clean.members
raw.till_totals ──────> clean.till_totals
```

Look at `clean.sales_lines`. It is built from three tables, and both outputs are built from it.

## Step 2: The people

A table does not get hurt. People do. Anything that reads a table is called its **consumer**: a person, a report, or another table.

These are the people who read Lotus's tables.

In [3]:
pd.DataFrame(consumers.CONSUMERS)[["who", "reads", "decides", "harm"]]

,who,reads,decides,harm
0,Store managers,[output.daily_store_sales],Nothing. They are paid from it: a weekly bonus of 1% of net sales,A wrong number is wrong pay
1,Head of retail operations,[output.daily_store_sales],Whether to sign off the bonus on Monday,"She signs a wrong number, and cannot say which one"
2,The replenishment team,[output.replenishment_list],How much of each product to order for tomorrow,Too high wastes food. Too low leaves empty shelves
3,Finance,"[output.replenishment_list, clean.supplier_prices]",How much to pay each supplier,A wrong price is wrong money leaving the company
4,Marketing,[clean.members],"Who gets an email, and at which address",One person stored twice gets two emails. A wrong address reaches a stranger


## Step 3: Follow one fault to the people

Suppose `stock_counts` is wrong tonight. Follow the arrows: which tables does the fault reach, and who reads those?

In [4]:
print(consumers.format_impact(consumers.impact("stock_counts")))

stock_counts: Counted by hand or by scanner.
  A wrong value here reaches these tables:
    clean.stock_counts
    output.replenishment_list
  and these people:
    The replenishment team
    Finance


That list is the **impact** of the fault: everything a wrong table touches. You will also hear it called the **blast radius**.

Now the opposite. Do the same for `till_totals`.

In [5]:
print(consumers.format_impact(consumers.impact("till_totals")))

till_totals: The till prints it when it is closed.
  A wrong value here reaches these tables:
    clean.till_totals
  and these people:
    nobody. No person reads it yet


Nobody. No output is built from it, and no person reads it.

## Step 4: Rank the seven tables by harm

Do it for all seven, and count the groups of people each one reaches.

In [6]:
ranked = pd.DataFrame(consumers.rank_tables())
ranked["groups reached"] = ranked["people"].str.len()
ranked[["rank", "table", "groups reached", "people", "critical"]]

,rank,table,groups reached,people,critical
0,1,sales_lines,4,"[Store managers, Head of retail operations, The replenishment team, Finance]",True
1,2,products,4,"[Store managers, Head of retail operations, The replenishment team, Finance]",True
2,3,stores,4,"[Store managers, Head of retail operations, The replenishment team, Finance]",True
3,4,stock_counts,2,"[The replenishment team, Finance]",False
4,5,supplier_prices,2,"[The replenishment team, Finance]",False
5,6,members,1,[Marketing],False
6,7,till_totals,0,[],False


Three tables reach four groups of people each. Those are Lotus's **critical tables**: a wrong value costs someone money or a decision. They get the most checks.

Two things in that list are worth a second look.

- **`stores` has six rows, and it is critical.** Every sales line is joined to it. A wrong row there is wrong everywhere. Size does not decide how much a table matters.
- **`till_totals` is last.** Nobody reads it. And in notebook 3 it was the only thing that proved the pipeline wrong. Keep that in mind: a table can be useless as an output and priceless for checking.

This map of "what reads what" has a name: **lineage**. Large teams use tools that draw it for them. You drew it by hand, from the SQL, and that is enough for seven tables.

In [7]:
sql("SELECT count(*) AS rows_in_stores FROM clean.stores")

,rows_in_stores
0,6


## What you now know

- A check protects a person. Ask who uses the table before you write one.
- A **consumer** is anything that reads a table.
- The **impact** of a fault is everything it touches: the tables, and the people who read them.
- A **critical table** is one where a wrong value costs someone money or a decision. Lotus has three.
- The ranking came from the readers, not from the size of the table.

**Say it in an interview:** "Before writing checks I ranked the seven tables by harm. I followed each one to the people who read it. Sales lines, products and stores each reached four groups, so they were critical. The stores table has six rows, and a wrong row there is wrong in every report."

**The theory:** [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), section "Lineage: who is affected". "Consumer" is defined in [Schema Evolution & Data Contracts](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/schema-evolution-and-data-contracts).

Next: [notebook 5](05_the_check_card.ipynb). Write the reasoning down.